# GDELT + ML + Black-Litterman 기반 동적 자산배분 파이프라인

### 구성 흐름
```
GDELT BigQuery (시장전체 + ETF별 + 개별종목별)
  ↓
yfinance (30종목 가격 데이터)
  ↓
패널 데이터 구성 (date × ticker × 피처)
  ↓
ML 앙상블 (RF / XGB / ET) - 롤링 윈도우
  ↓
Q, Ω 도출 → Black-Litterman
  ↓
MVO (γ별 5개 펀드 비중)
  ↓
백테스트 / 스트레스 테스트
```

### news_data.ipynb에서 이어받은 내용
- GDELT gdelt 패키지 구조 확인 (Events / Mentions / GKG 테이블)
- `parse_v2tone()` 함수 (V2Tone → 7차원 분해)
- GDELT BigQuery 쿼리 템플릿
- `classify_regime()` (레짐 분류)
- FinBERT 기반 감성 점수 계산 프레임워크

## 0. 패키지 설치 및 임포트

In [10]:
# !pip install google-cloud-bigquery pandas-gbq yfinance scikit-learn xgboost pandas numpy scipy

In [11]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
from scipy.optimize import minimize
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

# Google BigQuery
try:
    from google.cloud import bigquery
    BQ_AVAILABLE = True
    print("BigQuery 클라이언트 사용 가능")
except ImportError:
    BQ_AVAILABLE = False
    print("BigQuery 미설치 → 로컬 캐시 모드로 실행")

print("라이브러리 로드 완료")

BigQuery 클라이언트 사용 가능
라이브러리 로드 완료


## 1. 설정: 티커 리스트 & ETF/종목별 키워드 매핑

### 핵심 설계 원칙
- **모든 ETF와 개별종목**에 키워드를 지정하여 GDELT에서 해당 키워드가 포함된 기사의 감성을 별도 컬럼으로 수집
- 키워드는 GDELT GKG의 `Themes`, `Organizations`, `Locations` 필드와 매핑
- 각 ETF의 고유 감성 컬럼이 ML 피처로 직접 투입됨

In [12]:
# ============================================================
# 전체 30개 티커 (고정 순서 - BL 입력/출력 인덱스 일치)
# ============================================================
TICKERS = [
    # 인덱스 ETF (5)
    'SPY', 'QQQ', 'IWM', 'EFA', 'EEM',
    # 채권 ETF (4)
    'TLT', 'AGG', 'SHY', 'TIP',
    # 대안 ETF (2)
    'GLD', 'DBC',
    # 섹터 ETF (11)
    'XLK', 'XLF', 'XLE', 'XLV', 'VOX',
    'XLY', 'XLP', 'XLI', 'XLU', 'XLRE', 'XLB',
    # 개별종목 (8)
    'AAPL', 'MSFT', 'AMZN', 'GOOGL',
    'JPM',  'JNJ',  'PG',   'XOM'
]

# ============================================================
# ETF/종목별 GDELT 키워드 매핑
#   - 리스트 안의 키워드 중 하나라도 Themes/Organizations에 포함되면 매칭
#   - BigQuery WHERE절에서 OR로 연결됨
# ============================================================
TICKER_KEYWORDS = {
    # ── 인덱스 ETF ──────────────────────────────────────────
    'SPY':  ['ECON_STOCKMARKET', 'S&P 500', 'US equity', 'Wall Street', 'stock market rally'],
    'QQQ':  ['NASDAQ', 'tech stocks', 'growth stocks', 'big tech', 'technology rally'],
    'IWM':  ['small cap', 'Russell 2000', 'domestic economy', 'small business', 'SME'],
    'EFA':  ['developed markets', 'Europe stocks', 'MSCI EAFE', 'European equity', 'Japan stocks'],
    'EEM':  ['emerging markets', 'China stocks', 'EM equity', 'developing economies', 'BRIC'],

    # ── 채권 ETF ────────────────────────────────────────────
    'TLT':  ['treasury bonds', 'long-term bonds', 'ECON_FRBRESERVE', 'yield curve', '10-year treasury'],
    'AGG':  ['bond market', 'fixed income', 'investment grade', 'credit market', 'aggregate bond'],
    'SHY':  ['short-term treasury', 'T-bill', 'money market', '2-year treasury', 'short rates'],
    'TIP':  ['ECON_INFLATION', 'TIPS', 'CPI', 'inflation expectations', 'real yield'],

    # ── 대안 ETF ────────────────────────────────────────────
    'GLD':  ['gold', 'precious metals', 'safe haven', 'gold price', 'bullion'],
    'DBC':  ['commodities', 'oil price', 'raw materials', 'commodity index', 'energy prices'],

    # ── 섹터 ETF ────────────────────────────────────────────
    'XLK':  ['technology sector', 'semiconductor', 'artificial intelligence', 'cloud computing', 'chip stocks'],
    'XLF':  ['financial sector', 'banking', 'JPMorgan', 'interest rates', 'bank earnings'],
    'XLE':  ['energy sector', 'oil companies', 'natural gas', 'ExxonMobil', 'crude oil'],
    'XLV':  ['healthcare sector', 'pharmaceutical', 'biotech', 'drug approval', 'FDA'],
    'VOX':  ['communication services', 'media stocks', 'streaming', 'telecom', 'social media'],
    'XLY':  ['consumer discretionary', 'retail sales', 'consumer spending', 'e-commerce', 'luxury goods'],
    'XLP':  ['consumer staples', 'food stocks', 'household products', 'grocery', 'defensive stocks'],
    'XLI':  ['industrials', 'manufacturing', 'infrastructure', 'aerospace defense', 'transportation'],
    'XLU':  ['utilities', 'electricity', 'renewable energy', 'power grid', 'nuclear energy'],
    'XLRE': ['real estate', 'REIT', 'housing market', 'commercial real estate', 'property prices'],
    'XLB':  ['materials', 'mining', 'steel', 'chemicals', 'copper prices'],

    # ── 개별종목 ─────────────────────────────────────────────
    'AAPL':  ['Apple', 'iPhone', 'AAPL', 'Tim Cook', 'App Store'],
    'MSFT':  ['Microsoft', 'Azure', 'MSFT', 'Satya Nadella', 'Windows'],
    'AMZN':  ['Amazon', 'AWS', 'AMZN', 'Jeff Bezos', 'e-commerce giant'],
    'GOOGL': ['Google', 'Alphabet', 'GOOGL', 'YouTube', 'Google Cloud'],
    'JPM':   ['JPMorgan', 'Jamie Dimon', 'JPM', 'Chase bank', 'investment banking'],
    'JNJ':   ['Johnson Johnson', 'JNJ', 'Kenvue', 'pharmaceutical giant', 'medical devices'],
    'PG':    ['Procter Gamble', 'PG stock', 'consumer brands', 'Tide brand', 'P&G'],
    'XOM':   ['ExxonMobil', 'Exxon', 'XOM', 'oil major', 'fossil fuel'],
}

print(f"총 {len(TICKERS)}개 티커 설정 완료")
print(f"키워드 매핑: {len(TICKER_KEYWORDS)}개 티커 × 평균 {np.mean([len(v) for v in TICKER_KEYWORDS.values()]):.1f}개 키워드")

총 30개 티커 설정 완료
키워드 매핑: 30개 티커 × 평균 5.0개 키워드


## 2. GDELT BigQuery 쿼리 함수

### news_data.ipynb에서 확인한 구조 기반
- **GKG**: `V2Tone` (감성 7차원), `Themes`, `GCAM` (금융감성 c15)
- **MENTIONS**: `MentionDocTone`, `Confidence` → Ω proxy
- **EVENTS**: `GoldsteinScale`, `NumMentions` → 충격 강도

In [13]:
# ============================================================
# BigQuery 클라이언트 초기화
# 사전 준비:
#   1. gcloud auth application-default login
#   2. 또는 서비스 계정 키 경로 지정
# ============================================================
if BQ_AVAILABLE:
    client = bigquery.Client(project='finance-data-analysis-493308')  # project= 필요시 명시
    print("BigQuery 클라이언트 초기화 완료")
else:
    client = None
    print("BigQuery 미사용 - 캐시된 CSV 로드 예정")

BigQuery 클라이언트 초기화 완료


In [14]:
def fetch_gdelt_market(start_date: str, end_date: str) -> pd.DataFrame:
    """
    GDELT GKG에서 시장 전체 감성 일별 집계
    (ECON_STOCKMARKET / ECON_FRBRESERVE / ECON_INFLATION 테마)

    반환 컬럼:
      - avg_tone        : 일별 평균 감성
      - tone_std        : 감성 변동성 (불확실성 proxy)
      - fin_sentiment   : GCAM c15 금융 감성 (긍정 - 부정)
      - omega_proxy     : 기사 간 의견 불일치 (MENTIONS.MentionDocTone std)
      - article_count   : 일별 기사 수
      - crisis_signal   : 최솟값 (극단적 부정 이벤트 감지)
      - news_spike      : 기사 수 급증 더미 (상위 95% 이상)
    """
    start = start_date.replace('-', '')
    end   = end_date.replace('-', '')

    query_gkg = f"""
    SELECT
        PARSE_DATE('%Y%m%d', CAST(DATE AS STRING)) AS date,
        AVG(CAST(SPLIT(V2Tone, ',')[OFFSET(0)] AS FLOAT64))    AS avg_tone,
        STDDEV(CAST(SPLIT(V2Tone, ',')[OFFSET(0)] AS FLOAT64)) AS tone_std,
        AVG(
            COALESCE(CAST(REGEXP_EXTRACT(GCAM, r'c15\.1:([0-9.]+)') AS FLOAT64), 0) -
            COALESCE(CAST(REGEXP_EXTRACT(GCAM, r'c15\.2:([0-9.]+)') AS FLOAT64), 0)
        ) AS fin_sentiment,
        MIN(CAST(SPLIT(V2Tone, ',')[OFFSET(0)] AS FLOAT64))    AS tone_min,
        COUNT(*) AS article_count
    FROM `gdelt-bq.gdeltv2.gkg`
    WHERE DATE BETWEEN {start} AND {end}
      AND (
            Themes LIKE '%ECON_STOCKMARKET%'
         OR Themes LIKE '%ECON_FRBRESERVE%'
         OR Themes LIKE '%ECON_INFLATION%'
         OR Themes LIKE '%ECON_UNEMPLOYMENT%'
      )
    GROUP BY date
    ORDER BY date
    """

    query_mentions = f"""
    SELECT
        PARSE_DATE('%Y%m%d', CAST(CAST(MentionTimeDate/1000000 AS INT64) AS STRING)) AS date,
        STDDEV(MentionDocTone) AS mention_tone_std,
        AVG(Confidence)        AS avg_confidence
    FROM `gdelt-bq.gdeltv2.mentions`
    WHERE Confidence > 50
      AND MentionTimeDate BETWEEN {start}000000 AND {end}235959
    GROUP BY date
    ORDER BY date
    """

    query_events = f"""
    SELECT
        PARSE_DATE('%Y%m%d', CAST(SQLDATE AS STRING)) AS date,
        AVG(GoldsteinScale) AS avg_shock,
        MIN(GoldsteinScale) AS min_shock,
        SUM(NumMentions)    AS total_mentions
    FROM `gdelt-bq.gdeltv2.events`
    WHERE SQLDATE BETWEEN {start} AND {end}
    GROUP BY date
    ORDER BY date
    """

    df_gkg      = client.query(query_gkg).to_dataframe()
    df_mentions = client.query(query_mentions).to_dataframe()
    df_events   = client.query(query_events).to_dataframe()

    # 날짜 기준 병합 (각 테이블 독립 집계 후 merge)
    df = df_gkg \
        .merge(df_mentions.rename(columns={'mention_tone_std': 'omega_proxy'}),
               on='date', how='outer') \
        .merge(df_events, on='date', how='outer') \
        .sort_values('date')

    df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)

    # 파생 피처
    df['crisis_signal'] = df['tone_min'].fillna(0)
    threshold = df['article_count'].quantile(0.95)
    df['news_spike']    = (df['article_count'] > threshold).astype(int)

    df['sent_zscore'] = (
        df['avg_tone'] - df['avg_tone'].mean()
    ) / df['avg_tone'].std()

    return df.fillna(method='ffill').fillna(0)


print("fetch_gdelt_market() 정의 완료")

fetch_gdelt_market() 정의 완료


In [15]:
def fetch_gdelt_ticker_sentiment(
    ticker: str,
    keywords: list,
    start_date: str,
    end_date: str
) -> pd.DataFrame:
    """
    특정 티커(ETF 포함)의 키워드 기반 일별 감성 점수 수집

    - news_data.ipynb의 V2Tone 파싱 로직 활용
    - GDELT GKG Themes/Organizations 필드에서 키워드 매칭
    - 인덱스 ETF ~ 개별종목 모두 동일한 방식으로 처리

    반환 컬럼:
      - ticker_sentiment : 키워드 관련 기사 평균 감성
      - ticker_tone_std  : 감성 표준편차 (Ω 조정용)
      - ticker_article_n : 기사 수 (신뢰도 가중치)
    """
    start = start_date.replace('-', '')
    end   = end_date.replace('-', '')

    # 키워드 → BigQuery WHERE 조건 (OR 연결)
    # GDELT GKG의 Themes / Organizations 필드 모두 검색
    kw_conditions = " OR ".join([
        f"(Themes LIKE '%{kw}%' OR Organizations LIKE '%{kw}%')"
        for kw in keywords
    ])

    query = f"""
    SELECT
        PARSE_DATE('%Y%m%d', CAST(DATE AS STRING)) AS date,
        AVG(CAST(SPLIT(V2Tone, ',')[OFFSET(0)] AS FLOAT64))    AS ticker_sentiment,
        STDDEV(CAST(SPLIT(V2Tone, ',')[OFFSET(0)] AS FLOAT64)) AS ticker_tone_std,
        COUNT(*) AS ticker_article_n
    FROM `gdelt-bq.gdeltv2.gkg`
    WHERE DATE BETWEEN {start} AND {end}
      AND ({kw_conditions})
    GROUP BY date
    ORDER BY date
    """

    df = client.query(query).to_dataframe()
    df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
    df['ticker'] = ticker

    return df


def fetch_all_ticker_sentiments(
    ticker_keywords: dict,
    start_date: str,
    end_date: str
) -> pd.DataFrame:
    """
    전체 30개 티커 감성 점수 일괄 수집
    - ETF 22개 + 개별종목 8개 모두 동일 방식
    - 결과: date × ticker 멀티인덱스 DataFrame
    """
    results = []
    total = len(ticker_keywords)

    for i, (ticker, keywords) in enumerate(ticker_keywords.items(), 1):
        print(f"[{i:2d}/{total}] {ticker} 감성 쿼리 중...", end=' ')
        try:
            df = fetch_gdelt_ticker_sentiment(ticker, keywords, start_date, end_date)
            results.append(df)
            print(f"{len(df)}일치 수집")
        except Exception as e:
            print(f"실패: {e}")
            # 실패 시 빈 DataFrame 삽입
            results.append(pd.DataFrame(
                columns=['ticker_sentiment', 'ticker_tone_std', 'ticker_article_n', 'ticker']
            ))

    all_df = pd.concat(results).reset_index()
    return all_df.set_index(['date', 'ticker'])


print("fetch_gdelt_ticker_sentiment() / fetch_all_ticker_sentiments() 정의 완료")

fetch_gdelt_ticker_sentiment() / fetch_all_ticker_sentiments() 정의 완료


In [16]:
# ============================================================
# 실제 데이터 수집 실행
# BigQuery 미사용 시 캐시 CSV에서 로드
# ============================================================
START_DATE = '2016-01-01'
END_DATE   = '2025-12-31'

if BQ_AVAILABLE:
    print("[Step 1] GDELT 시장 전체 감성 수집 중...")
    gdelt_market_df = fetch_gdelt_market(START_DATE, END_DATE)
    gdelt_market_df.to_csv('gdelt_market.csv')
    print(f"완료: {gdelt_market_df.shape}")
    display(gdelt_market_df.head())
else:
    # 로컬 캐시 로드
    try:
        gdelt_market_df = pd.read_csv('gdelt_market.csv', index_col='date', parse_dates=True)
        print(f"캐시 로드 완료: {gdelt_market_df.shape}")
    except FileNotFoundError:
        print("캐시 없음 → 더미 데이터로 파이프라인 테스트")
        # 더미 데이터 생성 (구조 확인용)
        dates = pd.date_range(START_DATE, END_DATE, freq='B')
        gdelt_market_df = pd.DataFrame({
            'avg_tone':       np.random.normal(-0.5, 1.5, len(dates)),
            'tone_std':       np.random.uniform(1, 4, len(dates)),
            'fin_sentiment':  np.random.normal(0, 0.5, len(dates)),
            'omega_proxy':    np.random.uniform(0.5, 3, len(dates)),
            'article_count':  np.random.randint(50, 500, len(dates)),
            'crisis_signal':  np.random.normal(-3, 2, len(dates)),
            'news_spike':     np.random.randint(0, 2, len(dates)),
            'sent_zscore':    np.random.normal(0, 1, len(dates)),
            'avg_shock':      np.random.normal(0, 2, len(dates)),
        }, index=dates)
        gdelt_market_df.index.name = 'date'
        print(f"더미 데이터 생성: {gdelt_market_df.shape}")

[Step 1] GDELT 시장 전체 감성 수집 중...


Forbidden: 403 Quota exceeded: Your project exceeded quota for free query bytes scanned. For more information, see https://cloud.google.com/bigquery/docs/troubleshoot-quotas; reason: quotaExceeded, location: unbilled.analysis, message: Quota exceeded: Your project exceeded quota for free query bytes scanned. For more information, see https://cloud.google.com/bigquery/docs/troubleshoot-quotas

Location: US
Job ID: 0de4d091-074d-4fc1-af39-b4f32cb8bca8


In [ ]:
if BQ_AVAILABLE:
    print("[Step 2] 전체 30개 티커 감성 수집 중 (약 30번 BigQuery 쿼리)...")
    gdelt_ticker_df = fetch_all_ticker_sentiments(TICKER_KEYWORDS, START_DATE, END_DATE)
    gdelt_ticker_df.to_csv('gdelt_ticker_sentiment.csv')
    print(f"완료: {gdelt_ticker_df.shape}")
    display(gdelt_ticker_df.head(10))
else:
    try:
        gdelt_ticker_df = pd.read_csv(
            'gdelt_ticker_sentiment.csv',
            index_col=['date', 'ticker'], parse_dates=['date']
        )
        print(f"캐시 로드 완료: {gdelt_ticker_df.shape}")
    except FileNotFoundError:
        print("캐시 없음 → 더미 티커 감성 데이터 생성")
        dates = pd.date_range(START_DATE, END_DATE, freq='B')
        rows = []
        for t in TICKERS:
            for d in dates:
                rows.append({
                    'date': d, 'ticker': t,
                    'ticker_sentiment': np.random.normal(0, 1),
                    'ticker_tone_std':  np.random.uniform(0.5, 3),
                    'ticker_article_n': np.random.randint(0, 30),
                })
        gdelt_ticker_df = pd.DataFrame(rows).set_index(['date', 'ticker'])
        print(f"더미 데이터 생성: {gdelt_ticker_df.shape}")

## 3. yfinance 가격 데이터 수집

In [ ]:
print("yfinance 30종목 가격 데이터 수집 중...")
print("(auto_adjust=True → 분기/배당 조정 자동 적용)")

raw = yf.download(
    TICKERS,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=True
)

price_df = raw['Close'].dropna(how='all')

# ── XLRE 데이터 확인 (2015-10-07 상장) ─────────────────────
xlre_start = price_df['XLRE'].first_valid_index()
print(f"\nXLRE 첫 유효 데이터: {xlre_start}")
if xlre_start > pd.Timestamp('2016-01-01'):
    print("경고: XLRE 2016-01 이전 데이터 없음 → VNQ 대체 검토 필요")
else:
    print("XLRE 2016-01 데이터 확인 완료")

# ── JNJ Kenvue 분사 확인 (2023-08) ──────────────────────────
jnj_aug2023 = price_df['JNJ']['2023-07-01':'2023-09-30']
jnj_drop = jnj_aug2023.pct_change().min()
print(f"\nJNJ 2023-08 구간 최대 일간 낙폭: {jnj_drop:.2%}")
if abs(jnj_drop) < 0.05:
    print("adjusted price 처리 확인 완료 (급락 없음)")
else:
    print("경고: 비조정 가격일 수 있음 → auto_adjust 확인")

print(f"\n가격 데이터: {price_df.shape}")
print(f"기간: {price_df.index[0].date()} ~ {price_df.index[-1].date()}")
display(price_df.tail())

## 4. 최종 패널 데이터 구성

### 설계 원칙
- `date × ticker` 약 75,000행
- 모든 ETF에 개별 `ticker_sentiment` 컬럼 (시장 전체값 ≠ 종목별 키워드 감성)
- GICS 재분류 더미 (GOOGL/VOX 2018-09 구조 변화 반영)

In [ ]:
def build_final_panel(
    price_df: pd.DataFrame,
    gdelt_market_df: pd.DataFrame,
    gdelt_ticker_df: pd.DataFrame
) -> pd.DataFrame:
    """
    3개 데이터셋 → 최종 패널 (약 75,000행)

    피처 구성:
      ① yfinance 가격 기반 (종목별)
         - mom_20, mom_60          : 모멘텀 (20/60일 평균 수익률)
         - vol_20, vol_60          : 변동성 (20/60일 표준편차)
         - rel_spy                 : SPY 대비 초과수익률
         - ma_gap_20               : 20일 이동평균 대비 괴리율
      ② GDELT 시장 전체 (모든 종목 공통)
         - fin_sentiment, omega_proxy, crisis_signal, news_spike, sent_zscore
      ③ GDELT 티커별 키워드 감성 (ETF 22개 + 개별종목 8개 모두)
         - ticker_sentiment        : 키워드 매칭 기사 평균 감성
         - ticker_tone_std         : 키워드 기사 감성 표준편차
         - ticker_article_n        : 키워드 기사 수 (신뢰도)
         - ticker_sent_vs_market   : 종목 감성 - 시장 감성 (상대적 강도)
      ④ 구조적 변화 더미
         - gics_reclass_2018       : GOOGL/VOX 2018-09 GICS 재분류

    타깃: 30일 후 수익률 5분위 등급 (0~4)
    """
    ret = price_df.pct_change()
    records = []

    print(f"패널 구성 시작: {len(price_df.columns)}개 티커 × {len(price_df)}일")

    for i, ticker in enumerate(price_df.columns):
        if (i + 1) % 10 == 0:
            print(f"  [{i+1}/{len(price_df.columns)}] {ticker} 처리 중...")

        df = pd.DataFrame(index=price_df.index)
        df['ticker'] = ticker

        # ① yfinance 피처
        df['mom_20']    = ret[ticker].rolling(20).mean()
        df['mom_60']    = ret[ticker].rolling(60).mean()
        df['vol_20']    = ret[ticker].rolling(20).std()
        df['vol_60']    = ret[ticker].rolling(60).std()
        df['rel_spy']   = ret[ticker] - ret.get('SPY', pd.Series(0, index=ret.index))
        df['ma_gap_20'] = (
            price_df[ticker] / price_df[ticker].rolling(20).mean() - 1
        )

        # ② GDELT 시장 전체 피처 (날짜 기준 merge)
        market_cols = [
            'fin_sentiment', 'omega_proxy', 'crisis_signal',
            'news_spike', 'sent_zscore', 'avg_shock'
        ]
        available_cols = [c for c in market_cols if c in gdelt_market_df.columns]
        df = df.merge(
            gdelt_market_df[available_cols],
            left_index=True, right_index=True, how='left'
        )

        # ③ GDELT 티커별 키워드 감성 (ETF/개별종목 모두)
        if ticker in TICKER_KEYWORDS:
            try:
                ticker_sent = gdelt_ticker_df.xs(ticker, level='ticker')[
                    ['ticker_sentiment', 'ticker_tone_std', 'ticker_article_n']
                ]
                df = df.merge(ticker_sent, left_index=True, right_index=True, how='left')
            except KeyError:
                df['ticker_sentiment'] = np.nan
                df['ticker_tone_std']  = np.nan
                df['ticker_article_n'] = np.nan

            # 기사 없는 날 → 시장 전체 감성으로 대체
            df['ticker_sentiment'] = df['ticker_sentiment'].fillna(
                df.get('fin_sentiment', 0)
            )
            df['ticker_tone_std']  = df['ticker_tone_std'].fillna(
                df.get('omega_proxy', 1)
            )
            df['ticker_article_n'] = df['ticker_article_n'].fillna(0)
        else:
            df['ticker_sentiment'] = df.get('fin_sentiment', 0)
            df['ticker_tone_std']  = df.get('omega_proxy', 1)
            df['ticker_article_n'] = 0

        # 종목 감성 - 시장 감성 (상대적 강도 피처)
        df['ticker_sent_vs_market'] = (
            df['ticker_sentiment'] - df.get('fin_sentiment', 0)
        )

        # ④ GICS 재분류 더미 (GOOGL, VOX: 2018-09-21 구조 변화)
        df['gics_reclass_2018'] = (df.index >= '2018-09-21').astype(int)

        # 타깃: 30일 후 수익률 (shift(-30))
        future_ret = ret[ticker].shift(-30)
        df['future_ret'] = future_ret

        records.append(df)

    panel_df = pd.concat(records)

    # 타깃 변수: 전체 패널 기준 5분위 등급 (0~4)
    # (롤링 윈도우 내에서 재계산 필요 - 여기서는 전체 기준 초기화)
    valid_mask = panel_df['future_ret'].notna()
    panel_df.loc[valid_mask, 'target'] = pd.qcut(
        panel_df.loc[valid_mask, 'future_ret'],
        q=5, labels=[0, 1, 2, 3, 4]
    ).astype(int)

    panel_df = panel_df.dropna(
        subset=['mom_20', 'vol_20', 'fin_sentiment', 'ticker_sentiment', 'target']
    )

    print(f"\n패널 구성 완료: {panel_df.shape}")
    print(f"타깃 분포:\n{panel_df['target'].value_counts().sort_index()}")

    return panel_df


print("build_final_panel() 정의 완료")

In [ ]:
panel_df = build_final_panel(price_df, gdelt_market_df, gdelt_ticker_df)

print("\n패널 샘플 (첫 5행):")
display(panel_df.head())
print("\n컬럼 목록:")
print(panel_df.columns.tolist())

## 5. 피처 정의 및 One-Hot 인코딩

In [ ]:
# ticker One-Hot 인코딩
ticker_dummies = pd.get_dummies(panel_df['ticker'], prefix='ticker')
panel_df = pd.concat([panel_df, ticker_dummies], axis=1)

# 피처 컬럼 목록 (고정 순서)
BASE_FEATURES = [
    # 가격 기반
    'mom_20', 'mom_60', 'vol_20', 'vol_60', 'rel_spy', 'ma_gap_20',
    # GDELT 시장 전체
    'fin_sentiment', 'omega_proxy', 'crisis_signal', 'news_spike',
    'sent_zscore', 'avg_shock',
    # GDELT 티커별 (ETF 포함 전체 30개)
    'ticker_sentiment', 'ticker_tone_std', 'ticker_article_n',
    'ticker_sent_vs_market',
    # 구조적 변화 더미
    'gics_reclass_2018',
]

TICKER_DUMMY_COLS = [c for c in ticker_dummies.columns if c.startswith('ticker_')]
FEATURE_COLS = BASE_FEATURES + TICKER_DUMMY_COLS

# 누락 컬럼 0으로 채우기
for col in FEATURE_COLS:
    if col not in panel_df.columns:
        panel_df[col] = 0

panel_df[FEATURE_COLS] = panel_df[FEATURE_COLS].fillna(0)

print(f"총 피처 수: {len(FEATURE_COLS)}")
print(f"  - 가격 기반: 6")
print(f"  - GDELT 시장 전체: 6")
print(f"  - GDELT 티커별 감성: 4")
print(f"  - 구조 더미: 1")
print(f"  - ticker One-Hot: {len(TICKER_DUMMY_COLS)}")

## 6. ML 앙상블 모델: 롤링 윈도우 학습/예측

### 롤링 전략
- **학습 창**: 150 거래일 (약 7개월)
- **예측 창**: 30 거래일 (1개월)
- **모델**: RF + XGBoost + ExtraTrees → 앙상블
- **Q 도출**: `predict_proba() × quintile_means` → 예측 수익률
- **Ω 도출**: 3개 모델 예측값의 분산 → 불확실성

In [ ]:
def build_models():
    """RF / XGBoost / ExtraTrees 모델 초기화"""
    rf = RandomForestClassifier(
        n_estimators=200, max_depth=8,
        min_samples_leaf=10, n_jobs=-1,
        random_state=42
    )
    et = ExtraTreesClassifier(
        n_estimators=200, max_depth=8,
        min_samples_leaf=10, n_jobs=-1,
        random_state=42
    )
    xgb_model = xgb.XGBClassifier(
        n_estimators=200, max_depth=6,
        learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, use_label_encoder=False,
        eval_metric='mlogloss', n_jobs=-1,
        random_state=42
    )
    return [rf, et, xgb_model]


def rolling_train_predict(
    panel_df: pd.DataFrame,
    feature_cols: list,
    train_window: int = 150,
    pred_window:  int = 30
) -> pd.DataFrame:
    """
    150일 학습 → 30일 예측, 매월 롤링

    반환 컬럼:
      - Q     : 3개 모델 앙상블 예측 수익률 (quintile_means 가중 평균)
      - Omega : 3개 모델 예측값 분산 (모델 불일치 = 불확실성)
      - 기존 패널 컬럼 포함
    """
    dates   = panel_df.index.unique().sort_values()
    results = []
    models  = build_models()

    total_windows = (len(dates) - train_window - pred_window) // pred_window
    print(f"롤링 윈도우 총 {total_windows}회 예정")

    for window_i, i in enumerate(
        range(train_window, len(dates) - pred_window, pred_window)
    ):
        if window_i % 10 == 0:
            print(f"  [{window_i+1}/{total_windows}] 학습기간: "
                  f"{dates[i-train_window].date()} ~ {dates[i-1].date()} "
                  f"| 예측기간: {dates[i].date()} ~ {dates[i+pred_window-1].date()}")

        train_dates = dates[i - train_window : i]
        pred_dates  = dates[i : i + pred_window]

        train = panel_df[panel_df.index.isin(train_dates)].copy()
        pred  = panel_df[panel_df.index.isin(pred_dates)].copy()

        if len(train) < 100 or len(pred) == 0:
            continue

        X_train = train[feature_cols].values
        y_train = train['target'].astype(int).values
        X_pred  = pred[feature_cols].values

        # 학습 구간 내 5분위 평균 수익률 (분류 → 수익률 변환 키)
        quintile_means = (
            train.groupby('target')['future_ret'].mean()
            .reindex([0, 1, 2, 3, 4])
            .fillna(0).values
        )  # [-0.082, -0.021, +0.008, +0.031, +0.094] 형태

        # 각 모델 학습 및 예측 확률 수집
        all_pred_returns = []
        for model in models:
            model.fit(X_train, y_train)
            proba = model.predict_proba(X_pred)  # (n_samples, 5)

            # 클래스 순서 보장 (일부 구간에 특정 등급 없을 수 있음)
            n_classes = proba.shape[1]
            qm = quintile_means[:n_classes]

            pred_return = proba @ qm  # (n_samples,)
            all_pred_returns.append(pred_return)

        all_pred_returns = np.array(all_pred_returns)  # (3, n_samples)

        pred = pred.copy()
        pred['Q']     = all_pred_returns.mean(axis=0)  # 앙상블 평균
        pred['Omega'] = all_pred_returns.var(axis=0)   # 모델 간 분산
        results.append(pred)

    if not results:
        print("경고: 예측 결과 없음")
        return pd.DataFrame()

    result_df = pd.concat(results)
    print(f"\n롤링 예측 완료: {result_df.shape}")
    return result_df


print("rolling_train_predict() 정의 완료")

In [ ]:
# ============================================================
# 롤링 예측 실행
# ============================================================
print("ML 앙상블 롤링 예측 시작...")
result_df = rolling_train_predict(
    panel_df,
    feature_cols=FEATURE_COLS,
    train_window=150,
    pred_window=30
)

if len(result_df) > 0:
    print("\n예측 결과 샘플 (Q, Omega 컬럼):")
    display(result_df[['ticker', 'Q', 'Omega', 'future_ret']].head(20))
    print(f"\nQ 통계: 평균={result_df['Q'].mean():.4f}, 표준편차={result_df['Q'].std():.4f}")
    print(f"Omega 통계: 평균={result_df['Omega'].mean():.6f}, 표준편차={result_df['Omega'].std():.6f}")

## 7. 피처 중요도 분석 (GDELT 피처 유효성 검증)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'  # Mac 한글 폰트
matplotlib.rcParams['axes.unicode_minus'] = False

# 전체 데이터로 RF 1회 학습 → 피처 중요도 추출
if len(panel_df) > 1000:
    sample_panel = panel_df.dropna(subset=['target'])

    X = sample_panel[FEATURE_COLS].values
    y = sample_panel['target'].astype(int).values

    rf_full = RandomForestClassifier(
        n_estimators=100, max_depth=8,
        min_samples_leaf=10, n_jobs=-1, random_state=42
    )
    rf_full.fit(X, y)

    # 피처 중요도 (ticker dummy 제외, 인터프리터블 피처만)
    importance = pd.Series(
        rf_full.feature_importances_,
        index=FEATURE_COLS
    )
    importance_no_dummy = importance[BASE_FEATURES].sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(
        importance_no_dummy.index[::-1],
        importance_no_dummy.values[::-1],
        color=['#e74c3c' if 'gdelt' in c or 'sentiment' in c or 'omega' in c
                         or 'crisis' in c or 'news' in c or 'tone' in c
               else '#3498db'
               for c in importance_no_dummy.index[::-1]]
    )
    ax.set_xlabel('Feature Importance')
    ax.set_title('RF 피처 중요도\n(빨간색: GDELT 감성 피처 / 파란색: 가격 기반 피처)')

    # GDELT 피처 합산
    gdelt_feats = [f for f in BASE_FEATURES if any(
        kw in f for kw in ['sentiment', 'omega', 'crisis', 'news', 'tone', 'shock', 'zscore']
    )]
    gdelt_importance_sum = importance[gdelt_feats].sum()
    total_importance = importance[BASE_FEATURES].sum()

    ax.text(0.95, 0.02,
            f'GDELT 피처 기여도: {gdelt_importance_sum/total_importance:.1%}',
            transform=ax.transAxes, ha='right', fontsize=11,
            color='#e74c3c', fontweight='bold')

    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("\n피처 중요도 (상위 10개):")
    print(importance_no_dummy.head(10))
else:
    print("데이터 부족 - 피처 중요도 분석 생략")

## 8. Black-Litterman 모델

### 입력값 매핑
- **π (Prior)**: 리스크 패리티 기반 균형 수익률
- **Q (Views)**: ML 앙상블 예측 수익률
- **Ω**: 모델 불확실성 + GDELT 감성 기반 동적 조정

In [ ]:
def risk_parity_weights(Sigma: np.ndarray) -> np.ndarray:
    """변동성 역비례 비중 (ML 실패 시 변동성 큰 자산 방어)"""
    vols = np.sqrt(np.diag(Sigma))
    vols = np.where(vols < 1e-8, 1e-8, vols)  # 0 방어
    w = 1.0 / vols
    return w / w.sum()


def adjust_omega_with_gdelt(
    raw_omega: np.ndarray,
    gdelt_row: pd.Series,
    omega_proxy_mean: float
) -> np.ndarray:
    """
    GDELT 신호 기반 Ω 동적 조정

    조정 원칙:
      - 감성 극단치  → 불확실성 ×1.5 (시장 분위기가 극단적일수록 ML 신뢰 하락)
      - 의견 불일치  → 불확실성 ×1.3 (기사 간 방향 불일치 = 정보 혼란)
      - 뉴스 급증    → 불확실성 ×1.2 (뉴스 폭발 = 이벤트 발생 = 모델 불확실성↑)
      - 위기 복합 신호 → 불확실성 ×2.0 → μ가 π(Prior)에 강하게 수렴
    """
    multiplier = 1.0

    sent_zscore   = gdelt_row.get('sent_zscore', 0)
    omega_proxy   = gdelt_row.get('omega_proxy', 0)
    news_spike    = gdelt_row.get('news_spike', 0)
    crisis_signal = gdelt_row.get('crisis_signal', 0)

    if abs(sent_zscore) > 2:
        multiplier *= 1.5

    if omega_proxy > omega_proxy_mean * 1.5:
        multiplier *= 1.3

    if news_spike == 1:
        multiplier *= 1.2

    # 위기 복합: 극단적 부정 감성 + VIX 대리변수 조건
    if sent_zscore < -2 and crisis_signal < -5:
        multiplier *= 2.0

    return raw_omega * multiplier


def black_litterman(
    pi: np.ndarray,
    Sigma: np.ndarray,
    P: np.ndarray,
    Q: np.ndarray,
    Omega: np.ndarray,
    tau: float = 0.05
) -> np.ndarray:
    """
    μ = [(τΣ)⁻¹ + PᵀΩ⁻¹P]⁻¹ × [(τΣ)⁻¹π + PᵀΩ⁻¹Q]

    메커니즘:
      - Ω 작음 (ML 확신) → μ가 Q(ML뷰)에 가까워짐
      - Ω 큼 (ML 불확실) → μ가 π(Prior)에 수렴
      - 위기 날짜 → Ω ×3.0 → μ ≈ π (시장 균형 비중으로 후퇴)
    """
    M1  = np.linalg.inv(tau * Sigma + np.eye(len(Sigma)) * 1e-8)
    M2  = P.T @ np.linalg.inv(Omega + np.eye(len(Omega)) * 1e-8) @ P
    mu  = np.linalg.solve(
        M1 + M2,
        M1 @ pi + P.T @ np.linalg.inv(Omega + np.eye(len(Omega)) * 1e-8) @ Q
    )
    return mu


print("Black-Litterman 관련 함수 정의 완료")

In [ ]:
def extract_bl_inputs(
    result_df: pd.DataFrame,
    date: pd.Timestamp,
    tickers: list,
    gdelt_market_df: pd.DataFrame
) -> tuple:
    """
    특정 날짜의 ML 예측 결과 → BL 입력 형식 변환

    반환:
      Q_vec      : (30,) 예측 수익률
      Omega_diag : (30×30) 대각 행렬 (GDELT 동적 조정 적용)
    """
    day_df = result_df[result_df.index == date].copy()

    if len(day_df) == 0:
        # 해당 날짜 예측 없음 → 이전 유효 날짜 탐색
        available_dates = result_df.index.unique()
        prior_dates = available_dates[available_dates <= date]
        if len(prior_dates) == 0:
            return None, None
        date = prior_dates[-1]
        day_df = result_df[result_df.index == date].copy()

    day_df = day_df.set_index('ticker')

    # 티커 정렬 (항상 동일 순서 유지)
    available_tickers = [t for t in tickers if t in day_df.index]
    if len(available_tickers) < len(tickers):
        missing = set(tickers) - set(available_tickers)
        print(f"경고: 예측값 없는 티커 {missing}")

    Q_vec = day_df.loc[available_tickers, 'Q'].values
    raw_omega_vals = day_df.loc[available_tickers, 'Omega'].values

    # GDELT 기반 Ω 동적 조정
    gdelt_row = gdelt_market_df.loc[
        gdelt_market_df.index.asof(date)
    ] if hasattr(gdelt_market_df.index, 'asof') else gdelt_market_df.iloc[0]

    omega_proxy_mean = gdelt_market_df['omega_proxy'].mean() \
        if 'omega_proxy' in gdelt_market_df.columns else 1.0

    adjusted_omega = adjust_omega_with_gdelt(
        raw_omega_vals, gdelt_row, omega_proxy_mean
    )
    Omega_diag = np.diag(adjusted_omega)

    return Q_vec, Omega_diag, available_tickers


print("extract_bl_inputs() 정의 완료")

## 9. MVO: γ별 5개 펀드 비중 도출

In [ ]:
def mvo_optimize(
    mu: np.ndarray,
    Sigma: np.ndarray,
    gamma: float,
    max_weight: float = 0.30
) -> np.ndarray:
    """
    MVO 목적함수: max(μᵀw - γ/2 × wᵀΣw)
    제약: Σw = 1, 0 ≤ w ≤ 0.30
    """
    n = len(mu)

    def neg_utility(w):
        return -(w @ mu - gamma / 2 * w @ Sigma @ w)

    result = minimize(
        neg_utility,
        x0=np.ones(n) / n,
        method='SLSQP',
        bounds=[(0, max_weight)] * n,
        constraints=[{'type': 'eq', 'fun': lambda w: w.sum() - 1}],
        options={'ftol': 1e-9, 'maxiter': 500}
    )
    if not result.success:
        return np.ones(n) / n  # 최적화 실패 시 균등 배분
    return result.x


FUND_GAMMAS = {
    '초공격형 (γ=1)': 1,
    '공격형 (γ=2)':   2,
    '중립형 (γ=4)':   4,
    '보수형 (γ=8)':   8,
    '초보수형 (γ=16)': 16,
}


def build_fund_products(mu: np.ndarray, Sigma: np.ndarray, tickers: list) -> pd.DataFrame:
    """
    γ별 5개 펀드 비중 산출

    반환: 티커 × 펀드명 DataFrame
    """
    fund_weights = {}
    for name, gamma in FUND_GAMMAS.items():
        weights = mvo_optimize(mu, Sigma, gamma)
        fund_weights[name] = weights

    df = pd.DataFrame(fund_weights, index=tickers)
    return df


print("MVO 관련 함수 정의 완료")

In [ ]:
# ============================================================
# 특정 날짜 BL + MVO 실행 예시
# ============================================================
if len(result_df) > 0:
    # 예측 결과의 마지막 날짜 사용
    target_date = result_df.index.max()
    print(f"BL + MVO 실행 날짜: {target_date.date()}")

    # 공분산 행렬 (최근 252일 수익률)
    ret_matrix = price_df.pct_change().dropna()
    recent_ret = ret_matrix[
        ret_matrix.index <= target_date
    ].tail(252)

    # 사용 가능한 티커만 선택
    available_tickers = [t for t in TICKERS if t in recent_ret.columns]
    Sigma = recent_ret[available_tickers].cov().values * 252  # 연율화

    # Prior (리스크 패리티)
    w_mkt = risk_parity_weights(Sigma)
    delta = 2.5
    pi    = delta * Sigma @ w_mkt

    # BL 입력 추출 (GDELT 동적 Ω 포함)
    Q_vec, Omega_diag, bl_tickers = extract_bl_inputs(
        result_df, target_date, available_tickers, gdelt_market_df
    )

    if Q_vec is not None and len(bl_tickers) == len(available_tickers):
        # Black-Litterman 실행
        P   = np.eye(len(bl_tickers))
        mu  = black_litterman(pi, Sigma, P, Q_vec, Omega_diag, tau=0.05)

        # BL 결과 출력
        bl_result = pd.DataFrame({
            'π (Prior)':   pi,
            'Q (ML뷰)':     Q_vec,
            'μ (BL재합)':   mu,
        }, index=bl_tickers)
        bl_result['해석'] = bl_result.apply(
            lambda row: 'ML 강하게 반영' if abs(row['μ (BL재합)'] - row['π (Prior)']) > 0.01
            else 'Prior에 수렴', axis=1
        )
        print("\nBlack-Litterman 결과:")
        display(bl_result.round(4))

        # MVO 실행
        fund_df = build_fund_products(mu, Sigma, bl_tickers)
        print("\nγ별 펀드 비중 (%):")
        display((fund_df * 100).round(2))
    else:
        print("BL 입력값 부족 - 예측 결과 확인 필요")
else:
    print("롤링 예측 결과 없음 - 모델 실행 필요")

## 10. 월별 리밸런싱 백테스트

In [ ]:
def run_backtest(
    result_df: pd.DataFrame,
    price_df: pd.DataFrame,
    gdelt_market_df: pd.DataFrame,
    gamma: float = 4,
    rebal_cost: float = 0.001  # 0.1% 편도 거래비용
) -> pd.DataFrame:
    """
    월별 리밸런싱 백테스트

    반환: 날짜별 포트폴리오 수익률 시계열
    """
    ret_matrix = price_df.pct_change().fillna(0)

    # 리밸런싱 날짜 (예측 결과의 첫 날짜)
    rebal_dates = sorted(result_df.index.unique())[::30]  # 약 30일 간격

    portfolio_rets  = []
    prev_weights    = None
    current_weights = None

    print(f"백테스트 기간: {rebal_dates[0].date()} ~ {rebal_dates[-1].date()}")
    print(f"리밸런싱 횟수: {len(rebal_dates)}회")

    for i, rebal_date in enumerate(rebal_dates[:-1]):
        next_date = rebal_dates[i + 1]

        # 이번 기간 티커 공분산 (최근 252거래일)
        recent_ret = ret_matrix[
            ret_matrix.index <= rebal_date
        ].tail(252)
        available_tickers = [t for t in TICKERS if t in recent_ret.columns]

        try:
            Sigma = recent_ret[available_tickers].cov().values * 252
            w_mkt = risk_parity_weights(Sigma)
            pi    = 2.5 * Sigma @ w_mkt

            Q_vec, Omega_diag, bl_tickers = extract_bl_inputs(
                result_df, rebal_date, available_tickers, gdelt_market_df
            )

            if Q_vec is None or len(bl_tickers) != len(available_tickers):
                current_weights = w_mkt  # BL 실패 시 리스크 패리티 폴백
            else:
                P  = np.eye(len(bl_tickers))
                mu = black_litterman(pi, Sigma, P, Q_vec, Omega_diag)
                current_weights = mvo_optimize(mu, Sigma, gamma)
        except Exception as e:
            current_weights = np.ones(len(available_tickers)) / len(available_tickers)

        # 거래비용 차감 (이전 비중 대비 회전율 × 비용)
        if prev_weights is not None and len(prev_weights) == len(current_weights):
            turnover = np.abs(current_weights - prev_weights).sum() / 2
            tc = turnover * rebal_cost
        else:
            tc = 0

        # 보유 기간 수익률
        period_ret = ret_matrix[
            (ret_matrix.index > rebal_date) &
            (ret_matrix.index <= next_date)
        ][available_tickers]

        if len(period_ret) > 0:
            portfolio_daily_ret = (period_ret * current_weights).sum(axis=1)
            portfolio_daily_ret.iloc[0] -= tc  # 리밸런싱 날 거래비용 차감
            portfolio_rets.append(portfolio_daily_ret)

        prev_weights = current_weights

    if not portfolio_rets:
        return pd.Series(dtype=float, name='portfolio')

    return pd.concat(portfolio_rets).sort_index().rename('portfolio')


def compute_metrics(ret_series: pd.Series) -> dict:
    """백테스트 성과 지표 계산"""
    cumulative = (1 + ret_series).cumprod()
    total_days = len(ret_series)
    years       = total_days / 252

    cagr  = cumulative.iloc[-1] ** (1 / years) - 1
    vol   = ret_series.std() * np.sqrt(252)
    sharpe = (ret_series.mean() * 252 - 0.02) / vol  # rf=2% 가정

    rolling_max = cumulative.cummax()
    drawdown = (cumulative - rolling_max) / rolling_max
    mdd = drawdown.min()

    return {
        'CAGR':   f'{cagr:.2%}',
        'Sharpe': f'{sharpe:.2f}',
        'MDD':    f'{mdd:.2%}',
        'Vol':    f'{vol:.2%}',
        'Total Return': f'{cumulative.iloc[-1]-1:.2%}'
    }


print("백테스트 함수 정의 완료")

In [ ]:
if len(result_df) > 0:
    print("백테스트 실행 중 (γ=4 중립형)...")
    portfolio_ret = run_backtest(
        result_df, price_df, gdelt_market_df,
        gamma=4, rebal_cost=0.001
    )

    # 벤치마크 (SPY 수익률)
    spy_ret = price_df['SPY'].pct_change().reindex(portfolio_ret.index).fillna(0)

    if len(portfolio_ret) > 0:
        metrics_port = compute_metrics(portfolio_ret)
        metrics_spy  = compute_metrics(spy_ret)

        print("\n성과 비교:")
        metrics_df = pd.DataFrame({
            'BL-ML 포트폴리오': metrics_port,
            'SPY 벤치마크':     metrics_spy
        })
        display(metrics_df)

        # 누적수익률 차트
        fig, axes = plt.subplots(2, 1, figsize=(14, 8))

        # 누적 수익률
        ax1 = axes[0]
        (1 + portfolio_ret).cumprod().plot(ax=ax1, label='BL-ML 포트폴리오', linewidth=2)
        (1 + spy_ret).cumprod().plot(ax=ax1, label='SPY', linestyle='--', linewidth=1.5)
        ax1.set_title('누적 수익률 비교')
        ax1.set_ylabel('누적배수')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # 드로다운
        ax2 = axes[1]
        cum_port = (1 + portfolio_ret).cumprod()
        dd_port = (cum_port - cum_port.cummax()) / cum_port.cummax()
        dd_port.plot(ax=ax2, label='BL-ML 포트폴리오', color='red', linewidth=1.5)

        cum_spy = (1 + spy_ret).cumprod()
        dd_spy = (cum_spy - cum_spy.cummax()) / cum_spy.cummax()
        dd_spy.plot(ax=ax2, label='SPY', linestyle='--', color='gray', linewidth=1)

        # GDELT 스트레스 구간 표시 (sent_zscore < -2)
        if 'sent_zscore' in gdelt_market_df.columns:
            stress_days = gdelt_market_df[
                gdelt_market_df['sent_zscore'] < -2
            ].index
            for d in stress_days:
                if portfolio_ret.index.min() <= d <= portfolio_ret.index.max():
                    ax2.axvline(d, color='orange', alpha=0.3, linewidth=0.5)

        ax2.set_title('드로다운 (주황 수직선: GDELT 스트레스 구간)')
        ax2.set_ylabel('드로다운')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig('backtest_result.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    print("롤링 예측 결과 없음 - 백테스트 생략")

## 11. GDELT 스트레스 테스트

GDELT `sent_zscore < -2` 구간에서 포트폴리오 성과를 별도 분석

In [ ]:
if len(result_df) > 0 and len(portfolio_ret) > 0:
    if 'sent_zscore' in gdelt_market_df.columns:
        # 스트레스 구간 식별 (sent_zscore < -2)
        stress_mask = gdelt_market_df['sent_zscore'] < -2
        stress_dates = gdelt_market_df[stress_mask].index

        # 포트폴리오 수익률 인덱스와 교집합
        common_dates = portfolio_ret.index.intersection(stress_dates)
        normal_dates = portfolio_ret.index.difference(stress_dates)

        if len(common_dates) > 0:
            stress_ret = portfolio_ret.loc[common_dates]
            normal_ret = portfolio_ret.loc[normal_dates]
            spy_stress = spy_ret.loc[common_dates]

            stress_metrics = {
                '스트레스 구간 일수':       f'{len(common_dates)}일',
                '스트레스 구간 포트 평균 일수익': f'{stress_ret.mean():.4%}',
                '스트레스 구간 SPY 평균 일수익':  f'{spy_stress.mean():.4%}',
                '일반 구간 포트 평균 일수익':     f'{normal_ret.mean():.4%}',
                '스트레스 구간 포트 누적':        f'{(1+stress_ret).prod()-1:.2%}',
                '스트레스 구간 SPY 누적':         f'{(1+spy_stress).prod()-1:.2%}',
            }
            print("GDELT 기반 스트레스 테스트 결과:")
            for k, v in stress_metrics.items():
                print(f"  {k}: {v}")
        else:
            print("스트레스 구간과 백테스트 기간 겹치지 않음")
    else:
        print("sent_zscore 컬럼 없음 - GDELT 데이터 확인 필요")
else:
    print("백테스트 결과 없음")

## 12. γ별 전체 펀드 백테스트 비교

In [ ]:
if len(result_df) > 0:
    fig, ax = plt.subplots(figsize=(14, 6))
    all_metrics = {}

    linestyles = ['-', '--', '-.', ':', (0, (3, 1, 1, 1))]
    colors     = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db', '#9b59b6']

    for (fund_name, gamma), ls, color in zip(
        FUND_GAMMAS.items(), linestyles, colors
    ):
        try:
            ret = run_backtest(
                result_df, price_df, gdelt_market_df,
                gamma=gamma, rebal_cost=0.001
            )
            if len(ret) > 0:
                (1 + ret).cumprod().plot(
                    ax=ax, label=fund_name,
                    linestyle=ls, color=color, linewidth=2
                )
                all_metrics[fund_name] = compute_metrics(ret)
        except Exception as e:
            print(f"  {fund_name} 실패: {e}")

    # SPY 벤치마크
    (1 + spy_ret).cumprod().plot(
        ax=ax, label='SPY 벤치마크',
        linestyle='--', color='black', linewidth=1.5, alpha=0.7
    )

    ax.set_title('γ별 5개 펀드 누적 수익률 비교 (GDELT + BL + MVO)')
    ax.set_ylabel('누적배수')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('fund_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

    if all_metrics:
        print("\n전체 펀드 성과 비교:")
        display(pd.DataFrame(all_metrics).T)
else:
    print("롤링 예측 결과 없음")

## 13. 결과 저장 및 요약

In [ ]:
# 결과 파일 저장
if len(result_df) > 0:
    result_df[['ticker', 'Q', 'Omega', 'future_ret', 'target']].to_csv(
        'ml_predictions.csv'
    )
    print("ml_predictions.csv 저장 완료")

panel_df.to_csv('panel_data.csv')
print("panel_data.csv 저장 완료")

print("\n=" * 50)
print("파이프라인 실행 완료")
print("=" * 50)
print(f"  데이터 기간  : {START_DATE} ~ {END_DATE}")
print(f"  총 티커 수   : {len(TICKERS)}개")
print(f"  패널 행 수   : {len(panel_df):,}행")
print(f"  피처 수      : {len(FEATURE_COLS)}개")
print(f"    - GDELT 시장전체 : 6개")
print(f"    - GDELT 티커별   : 4개 (ETF 22개 + 개별종목 8개 모두 포함)")
print(f"    - 가격 기반      : 6개")
print(f"    - 구조 더미      : 1개")
print(f"    - ticker OHE     : {len(TICKER_DUMMY_COLS)}개")
if len(result_df) > 0:
    print(f"  롤링 예측 행 수  : {len(result_df):,}행")